In [1]:
import pandas as pd

# 1. 파일 불러오기 (경로는 필요에 따라 수정)
file_path = "./archive/2020 전국 기후.csv"

# 2. CSV 파일 읽기 (cp949 인코딩)
df = pd.read_csv(file_path, encoding='cp949')
df.columns = df.columns.str.strip()  # 혹시 모를 공백 제거

# 3. 일시 컬럼을 datetime 형식으로 변환
df['일시'] = pd.to_datetime(df['일시'], format='%Y.%m.%d %H:%M', errors='coerce')
df = df.dropna(subset=['일시'])

# 4. 평균 낼 수 있는 수치형 컬럼 지정
numeric_cols = ['기온(°C)', '습도(%)', '지면온도(°C)']
numeric_cols = [col for col in numeric_cols if col in df.columns]

# 5. 일시별로 평균 계산 (전국 평균)
df_avg = df.groupby('일시')[numeric_cols].mean().reset_index()

# 6. 결과 저장 (선택사항)
df_avg.to_csv("./archive/2020_climate_avg.csv", index=False, encoding='utf-8-sig')  # 로컬 저장

# 7. 출력 (선택사항)
print(df_avg.head())


                   일시    기온(°C)      습도(%)  지면온도(°C)
0 2020-01-01 01:00:00 -5.229474  53.694737 -3.950000
1 2020-01-01 02:00:00 -5.210526  55.400000 -3.927368
2 2020-01-01 03:00:00 -5.061053  57.105263 -3.767368
3 2020-01-01 04:00:00 -4.854737  59.410526 -3.546316
4 2020-01-01 05:00:00 -4.336842  61.052632 -3.205263


In [2]:
import pandas as pd
import os

# 처리할 연도 목록
years = ['2020', '2021', '2022', '2023']

# 평균을 낼 컬럼
numeric_cols = ['기온(°C)', '습도(%)', '지면온도(°C)']

for year in years:
    file_name = f"./archive/{year} 전국 기후.csv"
    if not os.path.exists(file_name):
        print(f"{file_name} 파일이 없습니다.")
        continue

    # 1. CSV 파일 읽기
    df = pd.read_csv(file_name, encoding='cp949')
    df.columns = df.columns.str.strip()

    # 2. 일시 컬럼 datetime 변환
    df['일시'] = pd.to_datetime(df['일시'], format='%Y.%m.%d %H:%M', errors='coerce')
    df = df.dropna(subset=['일시'])

    # 3. 존재하는 컬럼만 포함
    valid_cols = [col for col in numeric_cols if col in df.columns]

    # 4. 일시 기준 평균 계산
    df_avg = df.groupby('일시')[valid_cols].mean().reset_index()

    # 🔹 5. 소수점 첫째 자리로 반올림
    df_avg[valid_cols] = df_avg[valid_cols].round(1)

    # 6. 저장
    output_name = f"./archive/{year}_climate_avg.csv"
    df_avg.to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"{output_name} 저장 완료")


./archive/2020_climate_avg.csv 저장 완료
./archive/2021_climate_avg.csv 저장 완료
./archive/2022_climate_avg.csv 저장 완료
./archive/2023_climate_avg.csv 저장 완료


In [3]:
import pandas as pd

# 병합할 연도
years = ['2020', '2021', '2022', '2023']

# 연도별 파일을 읽어서 리스트에 저장
df_list = []
for year in years:
    file_name = f"./archive/{year}_climate_avg.csv"
    try:
        df = pd.read_csv(file_name, encoding='utf-8-sig')
        df['일시'] = pd.to_datetime(df['일시'])
        df_list.append(df)
    except Exception as e:
        print(f"{file_name} 불러오기 실패: {e}")

# 하나로 병합
merged_df = pd.concat(df_list, ignore_index=True)

# 시간 순 정렬
merged_df = merged_df.sort_values(by='일시').reset_index(drop=True)

# 저장
merged_df.to_csv("./archive/climate_2020_2023_merged.csv", index=False, encoding='utf-8-sig')
print("✅ climate_2020_2023_merged.csv 저장 완료")


✅ climate_2020_2023_merged.csv 저장 완료


In [9]:
import pandas as pd

# 1. 파일 경로와 인코딩
path = "./archive/2020_2022_ev_load.csv"
df = pd.read_csv(path, encoding="cp949")
df.columns = df.columns.str.strip()

# 2. long format으로 변환
df_long = df.melt(id_vars=["일자", "충전방식"], var_name="시간", value_name="충전량")

# 3. '일시' 컬럼 만들기 (일자 + 시간 → datetime)
df_long["일시"] = pd.to_datetime(df_long["일자"] + " " + df_long["시간"], format="%Y.%m.%d %H시", errors="coerce")
df_long = df_long.dropna(subset=["일시"])

# 4. '급속'과 '완속'을 열로 분리 (pivot)
df_pivot = df_long.pivot_table(index="일시", columns="충전방식", values="충전량", aggfunc="sum").reset_index()
df_pivot.columns.name = None  # 멀티 인덱스 제거
df_pivot = df_pivot.rename(columns={"급속": "급속 충전량(kWh)", "완속": "완속 충전량(kWh)"})

# 5. 시간순 정렬 및 저장 (선택)
df_pivot = df_pivot.sort_values("일시").reset_index(drop=True)
df_pivot.to_csv("ev_charging_2020_2022_processed.csv", index=False, encoding="utf-8-sig")

print("✅ 전처리 완료: ev_charging_2020_2022_processed.csv 저장됨")


✅ 전처리 완료: ev_charging_2020_2022_processed.csv 저장됨


In [12]:
import pandas as pd

# 1. 파일 경로와 인코딩
path = "./archive/2023_ev_load.csv"
df = pd.read_csv(path, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

# 2. long format으로 변환
df_long = df.melt(id_vars=["일자", "충전방식"], var_name="시간", value_name="충전량")

# 3. '일시' 컬럼 만들기 (일자 + 시간 → datetime)
df_long["일시"] = pd.to_datetime(df_long["일자"] + " " + df_long["시간"], format="%Y.%m.%d %H시", errors="coerce")
df_long = df_long.dropna(subset=["일시"])

# 4. '급속'과 '완속'을 열로 분리 (pivot)
df_pivot = df_long.pivot_table(index="일시", columns="충전방식", values="충전량", aggfunc="sum").reset_index()
df_pivot.columns.name = None  # 멀티 인덱스 제거
df_pivot = df_pivot.rename(columns={"급속": "급속 충전량(kWh)", "완속": "완속 충전량(kWh)"})

# 5. 시간순 정렬 및 저장 (선택)
df_pivot = df_pivot.sort_values("일시").reset_index(drop=True)
df_pivot.to_csv("ev_charging_2023_processed.csv", index=False, encoding="utf-8-sig")

print("✅ 전처리 완료: ev_charging_2023_processed.csv 저장됨")


✅ 전처리 완료: ev_charging_2023_processed.csv 저장됨


In [21]:
import pandas as pd

# 원본 파일 경로
file_path = "./archive/2020_demand.csv"

# 1행을 header로 인식하여 불러오기
df_full = pd.read_csv(file_path, encoding="cp949")

# 2557번째 인덱스부터 2922번째 인덱스까지 데이터만 추출 (2558~2923행)
df_trimmed = df_full.iloc[2556:2922].copy()  # 인덱스는 0부터 시작

# 저장
df_trimmed.to_csv("./archive/2020_demand_trimmed.csv", index=False, encoding="utf-8-sig")


In [22]:
import pandas as pd
import os

# 1. 원본 파일 경로 (EUC-KR 인코딩)
file_paths = {
    "2021": "./archive/2021_demand.csv",
    "2022": "./archive/2022_demand.csv",
    "2023": "./archive/2023_demand.csv"
}

# 2. 저장 경로 (UTF-8 인코딩)
save_paths = {
    "2021": "./archive/2021_demand_utf.csv",
    "2022": "./archive/2022_demand_utf.csv",
    "2023": "./archive/2023_demand_utf.csv"
}

# 3. 변환 및 저장
for year in ["2021", "2022", "2023"]:
    df = pd.read_csv(file_paths[year], encoding="euc-kr")
    df.to_csv(save_paths[year], index=False, encoding="utf-8-sig")


In [23]:
import pandas as pd

# 병합할 파일 리스트
file_list = [
    './archive/2020_demand_utf.csv',
    './archive/2021_demand_utf.csv',
    './archive/2022_demand_utf.csv',
    './archive/2023_demand_utf.csv'
]

# 각 파일을 읽고 리스트에 저장
dfs = [pd.read_csv(file, encoding='utf-8-sig') for file in file_list]

# 데이터 병합 (위에서 아래로 순차적으로)
merged_df = pd.concat(dfs, ignore_index=True)

# 병합된 데이터 저장
merged_df.to_csv('./archive/2020_2023_demand_merged.csv', index=False, encoding='utf-8-sig')


In [25]:
import pandas as pd

# EUC-KR로 저장된 파일 읽기
input_path = './archive/2020_2022_ev_load.csv'
output_path = './archive/2020_2022_ev_load_utf.csv'

# 파일 읽기
df = pd.read_csv(input_path, encoding='euc-kr')

# UTF-8로 저장
df.to_csv(output_path, index=False, encoding='utf-8-sig')


In [27]:
import pandas as pd

# EV 충전량 파일 경로
ev_files = [
    './archive/2020_2022_ev_load_utf.csv',
    './archive/2023_ev_load.csv'
]

# 파일 읽기 (둘 다 utf-8-sig 기준)
ev_dfs = [pd.read_csv(file, encoding='utf-8-sig') for file in ev_files]

# 병합
ev_merged = pd.concat(ev_dfs, ignore_index=True)

# 저장
ev_merged.to_csv('./archive/2020_2023_ev_load_merged.csv', index=False, encoding='utf-8-sig')


In [37]:
import pandas as pd

# 파일 경로
demand_path = "./archive/2020_2023_demand_merged.csv"
ev_path = "./archive/2020_2023_ev_load_merged.csv"
climate_path = "./archive/climate_2020_2023_merged.csv"

# 데이터 로드
demand = pd.read_csv(demand_path)
ev = pd.read_csv(ev_path)
climate = pd.read_csv(climate_path)

# =====================
# 전력 수요 데이터 처리
# =====================
demand_melted = demand.melt(id_vars=["날짜"], var_name="시간", value_name="전력수요량(MW)")

# 날짜 포맷 처리
demand_melted["날짜"] = pd.to_datetime(demand_melted["날짜"], format="%Y.%m.%d", errors="coerce")

# 24시 → 0시로 바꾸고 하루 추가
demand_melted.loc[demand_melted["시간"] == "24시", "시간"] = "00시"
demand_melted.loc[demand_melted["시간"] == "00시", "날짜"] += pd.Timedelta(days=1)

# 시간 숫자 추출 및 datetime 생성
demand_melted["시간"] = demand_melted["시간"].str.replace("시", "").astype(int)
demand_melted["datetime"] = demand_melted["날짜"] + pd.to_timedelta(demand_melted["시간"], unit="h")

# =====================
# 전기차 충전 데이터 처리
# =====================
ev_melted = ev.melt(id_vars=["날짜", "충전방식"], var_name="시간", value_name="충전량(kWh)")
ev_melted["날짜"] = pd.to_datetime(ev_melted["날짜"], format="%Y.%m.%d", errors="coerce")

ev_melted.loc[ev_melted["시간"] == "24시", "시간"] = "00시"
ev_melted.loc[ev_melted["시간"] == "00시", "날짜"] += pd.Timedelta(days=1)

ev_melted["시간"] = ev_melted["시간"].str.replace("시", "").astype(int)
ev_melted["datetime"] = ev_melted["날짜"] + pd.to_timedelta(ev_melted["시간"], unit="h")

# 피벗
ev_pivoted = ev_melted.pivot(index="datetime", columns="충전방식", values="충전량(kWh)").reset_index()
ev_pivoted.columns.name = None

# =====================
# 기후 데이터 처리
# =====================
climate["datetime"] = pd.to_datetime(climate["날짜"], format="%Y.%m.%d %H:%M", errors="coerce")
climate = climate.drop(columns=["날짜"])

# =====================
# 병합
# =====================
merged = pd.merge(demand_melted[["datetime", "전력수요량(MW)"]],
                  ev_pivoted, on="datetime", how="left")
merged = pd.merge(merged, climate, on="datetime", how="left")

# 정렬 및 저장
merged = merged.sort_values("datetime")
merged.to_csv("./archive/final_merged.csv", index=False, encoding="utf-8-sig")


각 feature간 상관관계 (pearson) 계산
모든 feature 다 계산해서 시각화 ㄱㄱ
LSTM,
SHAP 이용해보자 -> xAI : 어떤 변수가 크게 영향을 미쳤는지 알 수 있음 
LSTM은 Time SHAP 이용하는게 좋음

논문 목적 : 우리는 이런 데이터셋 만들었고 벤치마크 주었으니 너네도 한번 해봐라